In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "Asia/Shanghai")

# path definition
base_path = "/Volumes/dev/ecommerce"
users_path = f"{base_path}/bronze/users"
products_path = f"{base_path}/bronze/products"
events_path = f"{base_path}/bronze/events"

# configuration
NUM_USERS_INITIAL = 60000
NUM_PRODUCTS_INITIAL = 5000

cities = ["Auckland", "Wellington", "Christchurch", "Hamilton", "Dunedin"]

# generate users
def generate_users(num_users, start_id=1):
    city_array = F.array(*[F.lit(c) for c in cities])

    return (
        spark.range(start_id, start_id + num_users)
        .withColumn("user_id", F.concat(F.lit("user_"), F.col("id")))
        .withColumn("country", F.lit("NZ"))
        .withColumn("city", city_array[(F.rand() * len(cities)).cast("int")])
        .withColumn(
            "signup_time",
            F.from_unixtime(
                F.unix_timestamp() - (F.rand() * 365 * 86400).cast("int")
            ).cast("timestamp")
        )
        .withColumn("update_time", F.current_timestamp())
        .drop("id")
    )

# generate products
def generate_products(num_products, start_id=1):
    return (
        spark.range(start_id, start_id + num_products)
        .withColumn("product_id", F.concat(F.lit("product_"), F.col("id")))
        .withColumn(
            "category",
            F.when(F.col("id") % 5 == 0, "electronics")
            .when(F.col("id") % 5 == 1, "fashion")
            .when(F.col("id") % 5 == 2, "home")
            .when(F.col("id") % 5 == 3, "beauty")
            .otherwise("sports")
        )
        .withColumn("brand", F.concat(F.lit("brand_"), (F.rand() * 100).cast("int")))
        .withColumn("price", F.round(F.rand() * 500 + 20, 2))
        .withColumn(
            "created_at",
            F.from_unixtime(
                F.unix_timestamp() - (F.rand() * 365 * 86400).cast("int")
            ).cast("timestamp")
        )
        .drop("id")
    )

# generate events
def generate_initial_events(users_df, products_df):

    # join users & products
    base = (
        users_df
        .crossJoin(products_df.sample(0.01))  # control the number of events
    )

    base = base.withColumn(
        "days_since_signup",
        F.datediff(F.current_date(), F.to_date("signup_time"))
    )

    base = base.withColumn(
        "event_offset_days",
        (F.rand() * F.col("days_since_signup")).cast("int")
    )

    base = base.withColumn(
        "event_time",
        F.expr("timestampadd(DAY, event_offset_days, signup_time)")
    )

    base = base.withColumn(
        "event_time",
        F.when(F.col("event_time") < F.col("created_at"), F.col("created_at"))
        .otherwise(F.col("event_time"))
    )

    base = base.withColumn(
        "event_type",
        F.when(F.rand() < 0.75, "view")
         .when(F.rand() < 0.88, "add_to_cart")
         .when(F.rand() < 0.97, "purchase")
         .otherwise("login")
    )

    base = base.withColumn(
        "device",
        F.when(F.rand() < 0.7, "mobile").otherwise("web")
    )

    base = base.withColumn("event_date", F.to_date("event_time"))

    return base.select(
        "user_id",
        "product_id",
        "event_time",
        "event_date",
        "event_type",
        "device"
    )



users_df = generate_users(NUM_USERS_INITIAL, 1)
products_df = generate_products(NUM_PRODUCTS_INITIAL, 1)
events_df = generate_initial_events(users_df, products_df)

users_df.repartition(1).write.format("delta").mode("overwrite").save(users_path)
products_df.repartition(1).write.format("delta").mode("overwrite").save(products_path)
events_df.write.format("delta").mode("overwrite").partitionBy("event_date").save(events_path)
